In [12]:
import polars as pl
import pandas as pd
import datetime as dt
import gc # Thư viện dọn rác bộ nhớ của Python

# =====================================================================
# CẤU HÌNH ĐƯỜNG DẪN DỮ LIỆU
# =====================================================================
# Nhớ đổi lại đường dẫn này cho khớp với thư mục của bạn nhé!
file_path = 'Data/*.csv'

print("=====================================================")
print(" PHẦN 1 & 2: TIỀN XỬ LÝ VÀ DỌN RÁC (LAZY CLEAN)")
print("=====================================================")
print("Đang lên kế hoạch dọn dẹp dữ liệu...")

lazy_raw = pl.scan_csv(file_path, ignore_errors=True)

# 1. Xóa trùng lặp và lọc rác, giá âm
lazy_df = lazy_raw.unique()
lazy_df = lazy_df.filter(
    pl.col("user_session").is_not_null() & 
    pl.col("user_id").is_not_null() &
    (pl.col("price") > 0)
)

# 2. Bỏ cột rỗng, điền khuyết Brand
lazy_df = lazy_df.drop("category_code")
lazy_df = lazy_df.with_columns([
    pl.col("brand").fill_null("unknown_brand")
])

# 3. Chuẩn hóa thời gian
lazy_df = lazy_df.with_columns([
    pl.col("event_time")
    .str.slice(0, 19)
    .str.to_datetime(format="%Y-%m-%d %H:%M:%S")
    .alias("event_time")
]).with_columns([
    pl.col("event_time").dt.weekday().alias("day_of_week"),
    pl.col("event_time").dt.hour().alias("hour_of_day")
])


print("\n=====================================================")
print(" PHẦN 3: TÍNH TOÁN ĐẶC TRƯNG HỌC MÁY")
print("=====================================================")

# Khai báo tính RFM
lazy_rfm = (
    lazy_df.filter(pl.col("event_type") == "purchase")
    .group_by("user_id")
    .agg([
        pl.col("event_time").max().alias("last_purchase_date"),
        pl.col("user_session").n_unique().alias("Frequency"),
        pl.col("price").sum().alias("Monetary")
    ])
)

# Khai báo tính Session
lazy_session = (
    lazy_df.filter(pl.col("event_type") != "purchase")
    .group_by("user_session")
    .agg([
        pl.col("user_id").first().alias("user_id"),
        pl.col("day_of_week").first().alias("day_of_week"),
        pl.col("hour_of_day").first().alias("hour_of_day"),
        (pl.col("event_type") == "view").sum().alias("total_views"),
        (pl.col("event_type") == "cart").sum().alias("total_carts"),
        (pl.col("event_type") == "remove_from_cart").sum().alias("total_removes"),
        (pl.col("event_time").max() - pl.col("event_time").min()).dt.total_seconds().alias("session_duration"),
        pl.col("price").mean().alias("avg_price"),
        pl.col("price").max().alias("max_price"),
        pl.col("product_id").n_unique().alias("unique_products"),
        pl.col("brand").n_unique().alias("unique_brands"),
        pl.col("brand").mode().first().alias("top_brand")
    ])
)

print("Kích hoạt Streaming Engine để cày dữ liệu... (BƯỚC NÀY TỐN CPU VÀ RAM)")
# Thực thi và lấy bảng
rfm_table = lazy_rfm.collect(engine="streaming")
print(" Xong bảng RFM!")

session_table = lazy_session.collect(engine="streaming")
print("Xong bảng Session!")

purchased_sessions = (
    lazy_df.filter(pl.col("event_type") == "purchase")
    .select("user_session").unique().collect(engine="streaming")
)["user_session"].to_list()
print("Xong danh sách mua hàng!")

# Tính số ngày Recency
current_date = rfm_table["last_purchase_date"].max() + dt.timedelta(days=1)
rfm_table = rfm_table.with_columns([
    (current_date - pl.col("last_purchase_date")).dt.total_days().alias("Recency")
])


print("\n=====================================================")
print(" PHẦN 4: HỢP NHẤT BẰNG POLARS VÀ DỌN RÁC BỘ NHỚ")
print("=====================================================")
print("Đang gộp bảng bằng Polars để chống tràn RAM...")

# Gộp bảng ngay trong Polars
final_polars_table = session_table.join(
    rfm_table.select(['user_id', 'Frequency', 'Monetary', 'Recency']), 
    on='user_id', 
    how='left'
)

print("Đang dọn dẹp RAM...")
del rfm_table, session_table 
gc.collect() # Ép giải phóng RAM

print("Đang chuyển đổi dữ liệu an toàn sang Pandas...")
df_train = final_polars_table.to_pandas()

del final_polars_table
gc.collect()

print("Đang chấm điểm RFM và mã hóa...")
# Chấm điểm RFM
df_train['R_Score'] = pd.qcut(df_train['Recency'], q=4, labels=[4, 3, 2, 1]).astype(float).fillna(0).astype(int)
df_train['F_Score'] = pd.qcut(df_train['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(float).fillna(0).astype(int)
df_train['M_Score'] = pd.qcut(df_train['Monetary'], q=4, labels=[1, 2, 3, 4]).astype(float).fillna(0).astype(int)
df_train['RFM_Total_Score'] = df_train['R_Score'] + df_train['F_Score'] + df_train['M_Score']

# Gắn nhãn và cờ
df_train['is_purchased'] = df_train['user_session'].isin(purchased_sessions).astype(int)
df_train['is_single_brand'] = (df_train['unique_brands'] == 1).astype(int)

# Điền giá trị thiếu
cols_to_fill = ['R_Score', 'F_Score', 'M_Score', 'RFM_Total_Score', 'avg_price', 'max_price']
df_train[cols_to_fill] = df_train[cols_to_fill].fillna(0)

# Mã hóa Mục tiêu cho Brand (Target Encoding)
brand_target_mean = df_train.groupby('top_brand')['is_purchased'].mean()
df_train['top_brand_encoded'] = df_train['top_brand'].map(brand_target_mean)
df_train = df_train.drop(columns=['top_brand'])

display(df_train.head())

 PHẦN 1 & 2: TIỀN XỬ LÝ VÀ DỌN RÁC (LAZY CLEAN)
Đang lên kế hoạch dọn dẹp dữ liệu...

 PHẦN 3: TÍNH TOÁN ĐẶC TRƯNG HỌC MÁY
Kích hoạt Streaming Engine để cày dữ liệu... (BƯỚC NÀY TỐN CPU VÀ RAM)
 Xong bảng RFM!
Xong bảng Session!
Xong danh sách mua hàng!

 PHẦN 4: HỢP NHẤT BẰNG POLARS VÀ DỌN RÁC BỘ NHỚ
Đang gộp bảng bằng Polars để chống tràn RAM...
Đang dọn dẹp RAM...
Đang chuyển đổi dữ liệu an toàn sang Pandas...
Đang chấm điểm RFM và mã hóa...


,user_session,user_id,day_of_week,hour_of_day,total_views,total_carts,total_removes,session_duration,avg_price,max_price,...,Frequency,Monetary,Recency,R_Score,F_Score,M_Score,RFM_Total_Score,is_purchased,is_single_brand,top_brand_encoded
0,bfb175a9-8dc3-4d8f-b0f9-9e18be92105f,542222658,7,15,1,0,0,0,14.290000,14.29,...,NaN,NaN,NaN,0,0,0,0,0,1,0.038871
1,7a797423-4f92-4b80-ad7b-4bd47942b958,598810471,7,9,1,0,0,0,7.140000,7.14,...,NaN,NaN,NaN,0,0,0,0,0,1,0.029315
2,50fb3505-e5e7-4650-ba57-e02b9b4e8508,487361785,3,21,6,2,0,1292,5.485000,15.48,...,2.0,35.8,52.0,2,3,2,7,0,0,0.038871
3,6582b62f-049b-41a0-bf68-99605ca18711,469784640,6,15,1,0,0,0,13.130000,13.13,...,NaN,NaN,NaN,0,0,0,0,0,1,0.034522
4,16e69fd7-7f13-4714-b673-7de5d8445e78,580862178,5,9,7,0,0,545,3.728571,18.89,...,NaN,NaN,NaN,0,0,0,0,0,0,0.038871


In [13]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 4504473 entries, 0 to 4504472
Data columns (total 22 columns):
 #   Column             Dtype  
---  ------             -----  
 0   user_session       str    
 1   user_id            int64  
 2   day_of_week        int8   
 3   hour_of_day        int8   
 4   total_views        uint32 
 5   total_carts        uint32 
 6   total_removes      uint32 
 7   session_duration   int64  
 8   avg_price          float64
 9   max_price          float64
 10  unique_products    uint32 
 11  unique_brands      uint32 
 12  Frequency          float64
 13  Monetary           float64
 14  Recency            float64
 15  R_Score            int64  
 16  F_Score            int64  
 17  M_Score            int64  
 18  RFM_Total_Score    int64  
 19  is_purchased       int64  
 20  is_single_brand    int64  
 21  top_brand_encoded  float64
dtypes: float64(6), int64(8), int8(2), str(1), uint32(5)
memory usage: 764.7 MB


In [ ]:
# 1. Khai báo danh sách các cột "rác" cần loại bỏ
cols_to_drop = [ 'Frequency', 'Monetary', 'Recency']

# 2. Xóa các cột này khỏi tập dữ liệu
df_train = df_train.drop(columns=cols_to_drop)

In [15]:
df_train.head()

,user_session,user_id,day_of_week,hour_of_day,total_views,total_carts,total_removes,session_duration,avg_price,max_price,unique_products,unique_brands,R_Score,F_Score,M_Score,RFM_Total_Score,is_purchased,is_single_brand,top_brand_encoded
0,bfb175a9-8dc3-4d8f-b0f9-9e18be92105f,542222658,7,15,1,0,0,0,14.290000,14.29,1,1,0,0,0,0,0,1,0.038871
1,7a797423-4f92-4b80-ad7b-4bd47942b958,598810471,7,9,1,0,0,0,7.140000,7.14,1,1,0,0,0,0,0,1,0.029315
2,50fb3505-e5e7-4650-ba57-e02b9b4e8508,487361785,3,21,6,2,0,1292,5.485000,15.48,6,2,2,3,2,7,0,0,0.038871
3,6582b62f-049b-41a0-bf68-99605ca18711,469784640,6,15,1,0,0,0,13.130000,13.13,1,1,0,0,0,0,0,1,0.034522
4,16e69fd7-7f13-4714-b673-7de5d8445e78,580862178,5,9,7,0,0,545,3.728571,18.89,7,3,0,0,0,0,0,0,0.038871


In [16]:
df_train.describe()

,user_id,day_of_week,hour_of_day,total_views,total_carts,total_removes,session_duration,avg_price,max_price,unique_products,unique_brands,R_Score,F_Score,M_Score,RFM_Total_Score,is_purchased,is_single_brand,top_brand_encoded
count,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06,4.504473e+06
mean,5.373849e+08,3.910836e+00,1.277739e+01,2.137046e+00,1.245937e+00,6.590491e-01,1.535019e+04,1.382523e+01,1.640325e+01,2.927701e+00,1.504601e+00,7.721511e-01,7.681442e-01,7.680734e-01,2.308369e+00,3.103715e-02,8.061474e-01,3.103715e-02
std,8.032349e+07,1.975464e+00,5.358832e+00,5.489508e+00,5.898062e+00,4.771151e+00,3.085198e+05,2.726159e+01,3.133671e+01,7.675940e+00,1.457541e+00,1.315742e+00,1.309346e+00,1.309239e+00,3.770582e+00,1.734181e-01,3.953149e-01,9.192881e-03
min,4.654960e+05,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.000000e-02,5.000000e-02,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,5.159626e+08,2.000000e+00,9.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.130000e+00,3.650000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,2.539936e-02
50%,5.620748e+08,4.000000e+00,1.300000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,5.400000e+00,6.270000e+00,1.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00,3.082580e-02
75%,5.889452e+08,6.000000e+00,1.700000e+01,2.000000e+00,0.000000e+00,0.000000e+00,8.800000e+01,1.111000e+01,1.429000e+01,2.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,5.000000e+00,0.000000e+00,1.000000e+00,3.887120e-02
max,6.220902e+08,7.000000e+00,2.300000e+01,3.634000e+03,1.366000e+03,2.146000e+03,1.309956e+07,3.277800e+02,3.277800e+02,2.161000e+03,1.490000e+02,4.000000e+00,4.000000e+00,4.000000e+00,1.200000e+01,1.000000e+00,1.000000e+00,6.805075e-02


In [17]:
# Chỉ tính tương quan trên các cột số
correlation_matrix = df_train.corr(numeric_only=True)

# Xem độ tương quan của tất cả các biến đối với cột Mua Hàng (is_purchased)
# Sắp xếp từ cao xuống thấp để xem cái nào quan trọng nhất
print(correlation_matrix['is_purchased'].sort_values(ascending=False))

is_purchased         1.000000
unique_brands        0.258361
R_Score              0.232718
RFM_Total_Score      0.229255
F_Score              0.221024
total_carts          0.211233
M_Score              0.205334
unique_products      0.197230
total_views          0.159429
total_removes        0.152404
session_duration     0.055370
top_brand_encoded    0.053010
max_price            0.010425
hour_of_day         -0.002703
day_of_week         -0.003395
avg_price           -0.035914
user_id             -0.043390
is_single_brand     -0.243988
Name: is_purchased, dtype: float64


In [18]:
df_train.to_parquet('TrainData/Train.parquet')
print("Đã xuất file ")

Đã xuất file 
